# Análisis de Ventas — Superstore Dataset
Dataset: [Kaggle - Superstore Dataset](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final/data)  
Herramientas: Python, Pandas, SQLite, SQL

In [ ]:
import sqlite3
import pandas as pd

# Conectar a la base de datos
conn = sqlite3.connect('superstore.db')
print('Conexión exitosa a superstore.db')

## 1. Ventas y ganancia por categoría

In [ ]:
query = '''
SELECT 
    category,
    ROUND(SUM(sales), 2) AS total_ventas,
    ROUND(SUM(profit), 2) AS total_ganancia,
    ROUND(SUM(profit) * 100.0 / SUM(sales), 2) AS margen_pct
FROM orders
GROUP BY category
ORDER BY total_ventas DESC;
'''

df_categoria = pd.read_sql(query, conn)
df_categoria

**Hallazgo:** Technology lidera en ventas y tiene el mejor margen (17.4%). Furniture vende un volumen similar pero su margen es de apenas 2.49%.

## 2. Ventas y ganancia por sub-categoría

In [ ]:
query = '''
SELECT 
    category,
    sub_category,
    ROUND(SUM(sales), 2) AS total_ventas,
    ROUND(SUM(profit), 2) AS total_ganancia,
    ROUND(SUM(profit) * 100.0 / SUM(sales), 2) AS margen_pct
FROM orders
GROUP BY category, sub_category
ORDER BY margen_pct ASC;
'''

df_subcategoria = pd.read_sql(query, conn)
df_subcategoria

**Hallazgo:** Tres sub-categorías tienen ganancia negativa: Tables (-8.56%), Bookcases (-3.02%) y Supplies (-2.55%). Tables es el caso más crítico.

## 3. Descuento promedio vs ganancia por sub-categoría

In [ ]:
query = '''
SELECT 
    sub_category,
    ROUND(AVG(discount), 3) AS descuento_promedio,
    ROUND(SUM(profit), 2) AS ganancia_total,
    COUNT(*) AS num_ordenes
FROM orders
WHERE sub_category IN ('Tables', 'Bookcases', 'Supplies', 'Copiers', 'Paper')
GROUP BY sub_category
ORDER BY descuento_promedio DESC;
'''

df_descuento_sub = pd.read_sql(query, conn)
df_descuento_sub

**Hallazgo:** Tables y Bookcases tienen los descuentos promedio más altos (26% y 21%) y son las que más pierden. La hipótesis es que el descuento está destruyendo el margen.

## 4. Impacto del descuento en Tables por rango

In [ ]:
query = '''
SELECT 
    CASE 
        WHEN discount = 0 THEN '0%'
        WHEN discount <= 0.15 THEN '1-15%'
        WHEN discount <= 0.30 THEN '16-30%'
        ELSE '31%+'
    END AS rango_descuento,
    COUNT(*) AS num_ordenes,
    ROUND(SUM(sales), 2) AS ventas,
    ROUND(SUM(profit), 2) AS ganancia
FROM orders
WHERE sub_category = 'Tables'
GROUP BY rango_descuento
ORDER BY MIN(discount);
'''

df_tables = pd.read_sql(query, conn)
df_tables

**Hallazgo:** Con ventas casi idénticas en los tres rangos, la ganancia cae drásticamente a medida que sube el descuento. Tables sin descuento genera +$13.276, con descuento >30% genera -$27.296.

## 5. Ventas y ganancia por región

In [ ]:
query = '''
SELECT 
    region,
    ROUND(SUM(sales), 2) AS total_ventas,
    ROUND(SUM(profit), 2) AS total_ganancia,
    ROUND(SUM(profit) * 100.0 / SUM(sales), 2) AS margen_pct,
    COUNT(DISTINCT order_id) AS num_ordenes
FROM orders
GROUP BY region
ORDER BY total_ventas DESC;
'''

df_region = pd.read_sql(query, conn)
df_region

**Hallazgo:** West es la región más rentable (14.9% de margen). Central tiene el margen más bajo (7.92%) a pesar de vender más que South.

## 6. Descuento promedio por región

In [ ]:
query = '''
SELECT 
    region,
    ROUND(AVG(discount), 3) AS descuento_promedio,
    ROUND(SUM(profit), 2) AS ganancia_total
FROM orders
GROUP BY region
ORDER BY descuento_promedio DESC;
'''

df_descuento_region = pd.read_sql(query, conn)
df_descuento_region

**Hallazgo:** Patrón claro — a mayor descuento promedio, menor ganancia. Central tiene 24% de descuento promedio, casi el doble que West (10.9%).

## 7. Sub-categorías con peor desempeño en la región Central

In [ ]:
query = '''
SELECT 
    sub_category,
    ROUND(AVG(discount), 3) AS descuento_promedio,
    ROUND(SUM(profit), 2) AS ganancia
FROM orders
WHERE region = 'Central'
GROUP BY sub_category
ORDER BY ganancia ASC
LIMIT 5;
'''

df_central = pd.read_sql(query, conn)
df_central

**Hallazgo:** En Central, incluso sub-categorías rentables a nivel nacional (Appliances, Furnishings) son negativas por descuentos de entre 33% y 45%. El problema de Central es generalizado.

## 8. Evolución anual de ventas y ganancia

In [ ]:
query = '''
SELECT 
    strftime('%Y', order_date) AS anio,
    ROUND(SUM(sales), 2) AS total_ventas,
    ROUND(SUM(profit), 2) AS total_ganancia,
    ROUND(SUM(profit) * 100.0 / SUM(sales), 2) AS margen_pct
FROM orders
GROUP BY anio
ORDER BY anio;
'''

df_anual = pd.read_sql(query, conn)
df_anual

**Hallazgo:** Las ventas casi se duplicaron entre 2014 y 2017 ($484K → $733K), pero el margen se estanca entre 10-13%. Esto indica que los problemas de descuento son crónicos y no se han corregido en 4 años.

In [ ]:
# Cerrar la conexión al terminar
conn.close()
print('Conexión cerrada.')